# MoMo-FDVS logical PR13 — PaySim Version 2 registration

This notebook registers and validates the already-authorised PaySim Version 2 archive in private Drive storage. It performs no download, cannot access locked tests and cannot start training.

In [ ]:
RUN_PROFILE = "smoke"
TARGET_COMMIT = "9ac904bd9164a1c8848ad300addc1b2a89b7e144"
REPOSITORY_URL = "https://github.com/davidagyekum/momo-fraud-detection.git"
DRIVE_ROOT = "/content/drive/MyDrive/momo-fraud"
VM_ROOT = "/content/momo-work"
NOTEBOOK_PATH = "ml/notebooks/colab/02_dataset_acquisition_validation.ipynb"
ACTION = "register_paysim_v2"
assert RUN_PROFILE == "smoke"
assert ACTION == "register_paysim_v2"
assert len(TARGET_COMMIT) == 40 and TARGET_COMMIT != "REPLACE_WITH_PUSHED_PR13_SHA"

In [ ]:
from pathlib import Path
import subprocess
import sys
from google.colab import drive

drive.mount("/content/drive")
repo = Path(VM_ROOT) / "repo"
repo.parent.mkdir(parents=True, exist_ok=True)
if (repo / ".git").is_dir():
    subprocess.run(["git", "-C", str(repo), "fetch", "--prune", "origin"], check=True)
else:
    subprocess.run(["git", "clone", "--no-checkout", REPOSITORY_URL, str(repo)], check=True)
subprocess.run(["git", "-C", str(repo), "checkout", "--detach", TARGET_COMMIT], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "--requirement", str(repo / "ml/requirements-runtime.lock")], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "--no-deps", "--editable", str(repo / "ml")], check=True)
sys.path.insert(0, str(repo / "ml/src"))

In [ ]:
import json
from momo_fdvs_ml.acquisition import acquisition_readiness_report, register_local_source

report = acquisition_readiness_report(repo / "data")
sources = {source["dataset_id"]: source for source in report["sources"]}
assert sources["paysim"]["eligible_for_local_registration"] is True
assert report["network_acquisition_executed"] is False
assert report["training_executed"] is False

source_root = Path(DRIVE_ROOT) / "datasets"
source_path = source_root / "paysim-ealaxi-v2-f7eef9ffad5c.zip"
request_path = Path(VM_ROOT) / "outputs/paysim-v2-acquisition-request.json"
result_root = Path(DRIVE_ROOT) / "runs/paysim-registration-v2-f7eef9ff"
request_path.parent.mkdir(parents=True, exist_ok=True)
result_root.mkdir(parents=True, exist_ok=True)
request = {
    "schema_version": "acquisition-request-v1",
    "dataset_id": "paysim",
    "purpose": "internal_personal_noncommercial_academic_research",
    "reviewer_id": "REVIEWER_20260811A0B1",
    "permission_reference": "PERMISSION_20260811A0B1",
    "licence_reference": "LICENCE_CC4A20260811",
    "source_kind": "file",
    "source_path": str(source_path),
    "entrypoint": "PS_20174392719_1491204439457_log.csv",
    "expected_sha256": "f7eef9ffad5cfa64a034143a5c9b30491d189420b273d5ad5723ca40b596613d",
    "expected_size_bytes": 186385561,
    "expected_version": "kaggle-version-2-pending-byte-verification",
    "created_at": "2026-08-11T05:56:46Z",
    "acknowledgements": {
        "terms_reviewed": True,
        "no_redistribution": True,
        "private_storage": True,
        "version_verified": True,
    },
}
request_path.write_text(json.dumps(request, indent=2, sort_keys=True) + "\n", encoding="utf-8")
outputs = register_local_source(
    data_root=repo / "data",
    request_path=request_path,
    allowed_source_root=source_root,
    manifest_path=result_root / "dataset-registration-manifest.json",
    profile_path=result_root / "safe-profile.json",
)
safe_summary = {
    "dataset_id": outputs.manifest["dataset_id"],
    "status": outputs.manifest["status"],
    "source_sha256": outputs.manifest["source_sha256"],
    "source_size_bytes": outputs.manifest["source_size_bytes"],
    "file_count": outputs.manifest["file_count"],
    "inventory_sha256": outputs.manifest["inventory_sha256"],
    "validation_summary": outputs.manifest["validation_summary"],
    "quarantine_reasons": outputs.manifest["quarantine_reasons"],
    "network_acquisition_executed": outputs.manifest["network_acquisition_executed"],
    "source_bytes_committed": outputs.manifest["source_bytes_committed"],
    "promotable_for_training": outputs.manifest["promotable_for_training"],
}
print(json.dumps(safe_summary, indent=2, sort_keys=True))
assert outputs.manifest["status"] == "registered", outputs.manifest["quarantine_reasons"]
assert outputs.manifest["network_acquisition_executed"] is False
assert outputs.manifest["promotable_for_training"] is False

## Stop boundary

Stop after the PaySim registered/quarantined manifest and safe profile are written. Do not create splits, access locked tests, fit a model or promote this dataset; training remains a later owner-operated Colab milestone.